# Les décorateurs
## Décorateurs simples

In [1]:
def sandwich(garniture):
    def wrapper(*args, **kargs):
        print("/TTTTTTTT\\")
        price = garniture(*args, **kargs)
        print("\\________/")

        return price + 2

    return wrapper

In [2]:
@sandwich
def parisien(doux=False):
    print("  Jambon ")
    print("  Beurre  salé" if not doux else "  Beurre doux  ")

    return 6

@sandwich
def lyonnais(doux=False):
    print("  Rosette  ")
    print("  Beurre  salé" if not doux else "  Beurre doux  ")

    return 5

In [3]:
print(parisien(doux=True))

/TTTTTTTT\
  Jambon 
  Beurre doux  
\________/
8


In [4]:
lyonnais()

/TTTTTTTT\
  Rosette  
  Beurre  salé
\________/


7

## Décorateurs paramétrés

In [5]:
def tartine(_func=None, *, beurre=False):
    def deco(garniture):
        def wrapper():
            price = garniture()
            if beurre:
                print("  Beurre  ")
            print("\\________/")
    
            return price + 1
    
        return wrapper
    
    if _func is None:
        return deco
    else:
        return deco(_func)

    


In [6]:
@tartine
def confiture():
    print("  fraise  ")

    return 2

In [7]:
confiture()

  fraise  
\________/


3

## Exercices
### Exercice 1

In [8]:
import time

def time_this(func):
    def wrapper(*args, **kargs):
        start = time.time()

        value = func(*args, **kargs)
            
        end = time.time()
        print(end - start, "secondes se sont écoulées")
        
        return value

    return wrapper

In [9]:
@time_this
def long_run():
    for x in range(1_000_000):
        y = x ** 2
        
    return y

In [10]:
long_run()

0.03277707099914551 secondes se sont écoulées


999998000001

### Deuxième exercice

In [11]:
def count_calls(func):
    count = 0

    def wrapper(*args, **kargs):
        nonlocal count
        value = func()
        count += 1
        
        return value

    def get_count():
        return count

    wrapper.nbcalls = get_count

    return wrapper

In [12]:
@count_calls
def some_func():
    print("func called")

print(some_func.nbcalls())
some_func()
some_func()
print(some_func.nbcalls())


0
func called
func called
2


In [13]:
@count_calls
@time_this
def long_run():
    for x in range(1_000_000):
        y = x ** 2
        
    return y

In [14]:
return_value = long_run()

0.032631635665893555 secondes se sont écoulées


In [15]:
return_value

999998000001

In [16]:
long_run.nbcalls()

1

### Troisième question

In [17]:
import time
from collections import OrderedDict

def memoize(func):
    previous = OrderedDict()
    
    def inner(value:int):
        
        if value in previous:
            result = previous[value]
            previous.move_to_end(value)
        else:

            result = func(value)
    
            previous[value] = result

            if len(previous) > 3:
                previous.popitem(last=False)
            
        return result

    return inner

In [18]:
@time_this
@memoize
def long_call(value:int):
    time.sleep(2)
    return value**2


In [19]:
long_call(42)

2.003995180130005 secondes se sont écoulées


1764

In [20]:
long_call(52)

2.0044150352478027 secondes se sont écoulées


2704

In [21]:
long_call(65)

2.0002970695495605 secondes se sont écoulées


4225

In [22]:
long_call(value=5)

2.001646041870117 secondes se sont écoulées


25

### Quatrième question

In [23]:
notifications = []

def register(func):
    notifications.append(func)
    return func

@register
def notify_mail():
    print("notification on mail")

def notify_sms():
    print("notification on message")

@register
def notify_push():
    print("notification on push service")

def send_notifications():
    for notifiaction in notifications:
        notifiaction()

In [24]:
send_notifications()

notification on mail
notification on push service


## Décorateurs paramétrés

In [25]:
notifications = []

def register(_func=None, *, level=1):
    def deco_register(func):
        notifications.append((level, func))
        return func

    if _func is None:
        return deco_register
    else:
        return deco_register(_func)


@register(level=2)
def notify_mail():
    print("notification on mail")

@register
def notify_sms():
    print("notification on message")

@register(level=1)
def notify_push():
    print("notification on push service")

def send_notifications(level:int=1):
    for notif_level, notifiaction in notifications:
        if notif_level >= level:
            notifiaction()

In [26]:
send_notifications()

notification on mail
notification on message
notification on push service


In [27]:
send_notifications(level=2)

notification on mail
